# 01 · Teacher-forced evaluation on the Slakh test split

Reproduces Table 1 of the report (per-codebook cross-entropy on the 151 held-out Slakh test tracks) using only code from `src/`. It replaces `notebooks/archive/evaluation_test_slakh.ipynb` and must give the same numbers: the last cells assert this.

Compared with the archived notebook:
- models are rebuilt from each checkpoint's own config (the archived notebook built the Baseline from the OST-FT config; the architecture is identical, so results do not change);
- the unweighted average over codebooks is reported next to the weighted one;
- checkpoints use chronological names (Baseline → Slakh-FT → OST-FT) instead of CP1/CP2/CP3.

**Kaggle inputs to attach:** `slakh-tensors-final` and the three reported checkpoints.

In [ ]:
# ── Repository bootstrap: make src/ importable on Kaggle and locally ──
# On Kaggle the repository is cloned into /kaggle/working (Internet must be
# enabled in the notebook settings). Locally, set SECUNDA_REPO_DIR to the
# repository root and nothing is cloned.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DanBracs7/Adaptive-Secunda-Synth.git"
REPO_BRANCH = "revision"  # switch to "main" after merging the revision
REPO_DIR = Path(os.environ.get("SECUNDA_REPO_DIR", "/kaggle/working/Adaptive-Secunda-Synth"))

if not (REPO_DIR / "src").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR / "src"))

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()

print("Repository:", REPO_DIR)
print("Commit:", commit or "unknown (not a git checkout)")

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from audio_dataset import SecundaAudioDataset
from frame_codec import NUM_CODEBOOKS, CodebookSerializer
from paths import get_path, source_roots_for_split
from training_utils import evaluate_on_loader, load_checkpoint_model

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
FRAME_RATE = 75
SEGMENT_SECONDS = 10.0
BATCH_SIZE = 24
NUM_WORKERS = 2

# Weights of the "Weighted avg." row of Table 1. They are the OST
# fine-tune training weights and are applied to every checkpoint, so the
# weighted averages are comparable across checkpoints.
REPORT_LOSS_WEIGHTS = [1.0, 1.0, 2.0, 2.0, 2.5, 2.5, 2.5, 2.5]

# Chronological order: Baseline -> Slakh-FT -> OST-FT.
CHECKPOINTS = {
    "Baseline": get_path("ckpt_baseline"),
    "Slakh-FT": get_path("ckpt_slakh_ft"),
    "OST-FT": get_path("ckpt_ost_ft"),
}

OUTPUT_DIR = get_path("output_dir") / "eval_teacher_forced"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for label, path in CHECKPOINTS.items():
    assert path.exists(), f"{label}: {path}"

In [ ]:
test_dataset = SecundaAudioDataset(
    manifest_path=get_path("manifest"),
    source_roots=source_roots_for_split("test"),
    split="test",
    stage="pretrain",
    segment_seconds=SEGMENT_SECONDS,
    frame_rate=FRAME_RATE,
    fixed_start=True,
    seed=RANDOM_SEED,
    samples_per_track=1,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Slakh test tracks:", len(test_dataset.rows))
print("Test batches:", len(test_loader))

In [ ]:
serializer = CodebookSerializer()
results = {}

for label, path in CHECKPOINTS.items():
    model, checkpoint = load_checkpoint_model(path, device)

    metrics = evaluate_on_loader(
        model=model,
        loader=test_loader,
        serializer=serializer,
        device=device,
        codebook_loss_weights=REPORT_LOSS_WEIGHTS,
    )
    metrics["unweighted_loss"] = float(np.mean(metrics["per_codebook_loss"]))
    metrics["training_loss_weights"] = checkpoint.get("config", {}).get("codebook_loss_weights")
    results[label] = metrics

    print(f"── {label} ──")
    print("Weighted test loss (report weights):", metrics["loss"])
    print("Unweighted test loss:", metrics["unweighted_loss"])
    print("Per-codebook loss:", metrics["per_codebook_loss"])
    print("Per-codebook accuracy:", metrics["per_codebook_accuracy"])
    print("Training loss weights of this run:", metrics["training_loss_weights"])

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
codebook_labels = [f"Q{q}" for q in range(NUM_CODEBOOKS)]

loss_table = pd.DataFrame(
    {label: metrics["per_codebook_loss"] for label, metrics in results.items()},
    index=codebook_labels,
)
loss_table.loc["Weighted avg."] = [metrics["loss"] for metrics in results.values()]
loss_table.loc["Unweighted avg."] = [metrics["unweighted_loss"] for metrics in results.values()]

accuracy_table = pd.DataFrame(
    {label: metrics["per_codebook_accuracy"] for label, metrics in results.items()},
    index=codebook_labels,
)

print("Test cross-entropy (Table 1):")
print(loss_table.round(2).to_string())
print("\nTest token accuracy:")
print(accuracy_table.round(3).to_string())

loss_table.to_csv(OUTPUT_DIR / "table1_test_loss.csv")
accuracy_table.to_csv(OUTPUT_DIR / "test_accuracy.csv")
with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as file:
    json.dump(
        {"report_loss_weights": REPORT_LOSS_WEIGHTS, "results": results},
        file,
        indent=2,
    )

### Regression check against the first submission

The weighted losses must match the values printed by `notebooks/archive/evaluation_test_slakh.ipynb`. The tolerance allows for floating-point differences between GPU runs.

In [ ]:
FIRST_SUBMISSION_WEIGHTED_LOSS = {
    "Baseline": 5.540221895490374,
    "Slakh-FT": 5.253845146724156,
    "OST-FT": 5.322701181684222,
}
TOLERANCE = 1e-3

if len(test_dataset.rows) == 151:
    for label, expected in FIRST_SUBMISSION_WEIGHTED_LOSS.items():
        observed = results[label]["loss"]
        difference = abs(observed - expected)
        print(f"{label:9s} observed={observed:.6f} expected={expected:.6f} |diff|={difference:.2e}")
        assert difference < TOLERANCE, f"{label}: {observed} vs {expected}"
    print("PASS: Table 1 reproduced with the refactored code.")
else:
    print(f"Skipped: {len(test_dataset.rows)} test tracks, not the full 151-track split.")

In [ ]:
x = np.arange(NUM_CODEBOOKS)
width = 0.27

figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for offset, label in zip((-width, 0.0, width), results):
    axes[0].bar(x + offset, results[label]["per_codebook_loss"], width, label=label)
    axes[1].bar(x + offset, results[label]["per_codebook_accuracy"], width, label=label)

axes[0].set_ylabel("Cross-entropy")
axes[0].set_title("Per-codebook test loss (151 Slakh tracks)")
axes[1].set_ylabel("Token accuracy")
axes[1].set_title("Per-codebook test accuracy (151 Slakh tracks)")

for axis in axes:
    axis.set_xticks(x)
    axis.set_xticklabels(codebook_labels)
    axis.legend()

figure.tight_layout()
figure.savefig(OUTPUT_DIR / "per_codebook_test_metrics.png", dpi=200)
plt.show()